# Aula 04 — Teorema de Bayes e atualização de crenças

## Objetivo

Implementar atualizações bayesianas auditáveis, conferir o resultado por frequências naturais, simular o mecanismo, atualizar odds sequencialmente e analisar sensibilidade a priors e taxas de falso positivo.

> Este notebook complementa a Aula 04 do módulo **Probabilidade, Estatística e Teoria da Informação**.

## Premissas e limites

- O sistema de triagem é sintético e serve apenas para aprendizagem matemática.
- O cenário binário usa $P(H)=0,01$, sensibilidade 0,95 e taxa de falso positivo 0,05.
- As taxas do modelo são tratadas como conhecidas; aplicações reais precisam estimá-las e quantificar incerteza.
- Atualizações repetidas multiplicam razões de verossimilhança somente sob a fatoração condicional declarada.
- A seed fixa reproduz esta simulação na versão usada, mas não é evidência sobre o fenômeno.
- O posterior é condicional ao modelo, às hipóteses e à população escolhida.

## 1. Preparação

O Google Colab já inclui NumPy e Matplotlib. Em ambiente local:

```bash
python -m pip install "numpy>=1.24" "matplotlib>=3.7" jupyter
```

In [ ]:
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

SEED = 42
TOL = 0.005
rng = np.random.default_rng(SEED)

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Seed:", SEED)
print("Tolerância empírica:", TOL)

## 2. Bayes binário com validações

Para hipótese $H$ e evidência positiva:

$$
P(H\mid +)=\frac{P(+\mid H)P(H)}{P(+\mid H)P(H)+P(+\mid H^c)P(H^c)}.
$$

In [ ]:
def bayes_binario(prior, sensibilidade, taxa_falso_positivo):
    valores = (prior, sensibilidade, taxa_falso_positivo)
    if not all(np.isfinite(x) and 0 <= x <= 1 for x in valores):
        raise ValueError("Todas as probabilidades devem ser finitas e pertencer a [0, 1].")

    numerador = sensibilidade * prior
    evidencia = numerador + taxa_falso_positivo * (1 - prior)
    if evidencia == 0:
        raise ZeroDivisionError("A evidência observada tem probabilidade zero no modelo.")
    return numerador / evidencia, evidencia

posterior, evidencia = bayes_binario(0.01, 0.95, 0.05)
print(f"P(+)       = {evidencia:.6f}")
print(f"P(H | +)   = {posterior:.6f} ({100*posterior:.2f}%)")

assert np.isclose(evidencia, 0.059)
assert np.isclose(posterior, 95 / 590)

try:
    bayes_binario(0.5, 0.0, 0.0)
except ZeroDivisionError as erro:
    print("Erro esperado:", erro)

## 3. Frequências naturais

Em uma população de 10.000 itens, calculamos verdadeiros positivos e falsos positivos. Como os parâmetros produzem contagens inteiras neste exemplo, a tabela verifica exatamente a fórmula.

In [ ]:
N = 10_000
prior = 0.01
sensibilidade = 0.95
fpr = 0.05

n_h = round(N * prior)
n_nao_h = N - n_h
verdadeiros_positivos = round(n_h * sensibilidade)
falsos_negativos = n_h - verdadeiros_positivos
falsos_positivos = round(n_nao_h * fpr)
verdadeiros_negativos = n_nao_h - falsos_positivos

tabela = np.array([
    [verdadeiros_positivos, falsos_negativos],
    [falsos_positivos, verdadeiros_negativos],
])

print("[[VP, FN], [FP, VN]]")
print(tabela)
posterior_frequencias = verdadeiros_positivos / tabela[:, 0].sum()
print(f"P(H | +) por frequências = {posterior_frequencias:.6f}")

assert tabela.tolist() == [[95, 5], [495, 9405]]
assert np.isclose(posterior_frequencias, posterior)

## 4. Simulação reproduzível

Simulamos primeiro o estado real. Depois, a probabilidade do alerta depende desse estado. A estimativa deve ficar próxima do posterior teórico, dentro da tolerância declarada antes da execução.

In [ ]:
N_SIMULACOES = 500_000

h = rng.random(N_SIMULACOES) < prior
p_alerta = np.where(h, sensibilidade, fpr)
alerta = rng.random(N_SIMULACOES) < p_alerta

est_prior = h.mean()
est_evidencia = alerta.mean()
est_posterior = (h & alerta).sum() / alerta.sum()

print(f"Prior teórico / simulado:     {prior:.6f} / {est_prior:.6f}")
print(f"Evidence teórica / simulada:  {evidencia:.6f} / {est_evidencia:.6f}")
print(f"Posterior teórico / simulado: {posterior:.6f} / {est_posterior:.6f}")

assert abs(est_prior - prior) < TOL
assert abs(est_evidencia - evidencia) < TOL
assert abs(est_posterior - posterior) < TOL
print("Simulação compatível com o modelo na tolerância declarada.")

## 5. Sensibilidade ao prior

Mantemos sensibilidade 0,95 e falso positivo 0,05. O mesmo alerta produz posteriors muito diferentes conforme a prevalência anterior.

In [ ]:
priors = np.geomspace(0.0001, 0.5, 400)
posteriors = np.array([
    bayes_binario(p, sensibilidade, fpr)[0]
    for p in priors
])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(priors * 100, posteriors * 100, color="#16b8f3", linewidth=2.5)
ax.axvline(prior * 100, color="#ffb347", linestyle="--", label="prior do exemplo = 1%")
ax.scatter([prior * 100], [posterior * 100], color="#ff6b6b", zorder=3)
ax.set_xscale("log")
ax.set_xlabel("prior P(H), em % — escala log")
ax.set_ylabel("posterior P(H|+), em %")
ax.set_title("O posterior depende da taxa-base")
ax.grid(alpha=0.25, which="both")
ax.legend()
plt.show()

for p in [0.001, 0.01, 0.05, 0.10, 0.50]:
    post, _ = bayes_binario(p, sensibilidade, fpr)
    print(f"prior={100*p:5.1f}% → posterior={100*post:6.2f}%")

assert np.isclose(bayes_binario(0.05, 0.95, 0.05)[0], 0.5)

## 6. Mapa de sensibilidade ao prior e ao falso positivo

Cada célula mostra $P(H\mid +)$. Reduzir falso positivo pode ser decisivo quando o evento é raro.

In [ ]:
priors_grade = np.array([0.001, 0.005, 0.01, 0.05, 0.10])
fprs_grade = np.array([0.001, 0.005, 0.01, 0.05, 0.10])
mapa = np.empty((len(fprs_grade), len(priors_grade)))

for i, taxa_fp in enumerate(fprs_grade):
    for j, p in enumerate(priors_grade):
        mapa[i, j] = bayes_binario(p, sensibilidade, taxa_fp)[0]

fig, ax = plt.subplots(figsize=(8, 5.5))
imagem = ax.imshow(mapa, vmin=0, vmax=1, cmap="Blues", aspect="auto")
for i in range(mapa.shape[0]):
    for j in range(mapa.shape[1]):
        ax.text(j, i, f"{100*mapa[i, j]:.1f}%", ha="center", va="center",
                color="white" if mapa[i, j] > 0.55 else "#102a43")
ax.set_xticks(range(len(priors_grade)), labels=[f"{100*p:g}%" for p in priors_grade])
ax.set_yticks(range(len(fprs_grade)), labels=[f"{100*p:g}%" for p in fprs_grade])
ax.set_xlabel("prior P(H)")
ax.set_ylabel("taxa de falso positivo P(+|Hᶜ)")
ax.set_title("Posterior após um alerta — sensibilidade fixa em 95%")
fig.colorbar(imagem, ax=ax, label="P(H|+)")
plt.show()

assert np.all((mapa >= 0) & (mapa <= 1))

## 7. Atualização em odds

As odds posteriores são as odds anteriores multiplicadas pela razão de verossimilhança. Repetir o LR exige evidências condicionalmente independentes sob $H$ e $H^c$.

In [ ]:
def prob_para_odds(p):
    if not 0 < p < 1:
        raise ValueError("A conversão finita exige 0 < p < 1.")
    return p / (1 - p)

def odds_para_prob(o):
    if not np.isfinite(o) or o < 0:
        raise ValueError("Odds devem ser finitas e não negativas.")
    return o / (1 + o)

odds_prior = prob_para_odds(prior)
lr_positivo = sensibilidade / fpr
odds_apos_1 = odds_prior * lr_positivo
odds_apos_2 = odds_apos_1 * lr_positivo
p_apos_1 = odds_para_prob(odds_apos_1)
p_apos_2 = odds_para_prob(odds_apos_2)

print(f"Odds anteriores:           {odds_prior:.6f}")
print(f"LR positivo:               {lr_positivo:.2f}")
print(f"Posterior após 1 positivo: {p_apos_1:.6f}")
print(f"Posterior após 2 positivos:{p_apos_2:.6f}")

assert np.isclose(p_apos_1, posterior)
assert np.isclose(p_apos_2, 361 / 460)

print("A segunda atualização só é válida sob a independência condicional declarada.")

## 8. Múltiplas hipóteses

Calculamos scores $s_i=P(D\mid H_i)P(H_i)$ e os normalizamos. As validações impedem probabilidades negativas, vetores incompatíveis e evidence nula.

In [ ]:
def atualizar_multiplas(priors, likelihoods):
    priors = np.asarray(priors, dtype=float)
    likelihoods = np.asarray(likelihoods, dtype=float)
    if priors.ndim != 1 or likelihoods.shape != priors.shape:
        raise ValueError("Priors e likelihoods devem ser vetores 1D do mesmo tamanho.")
    if np.any(~np.isfinite(priors)) or np.any(~np.isfinite(likelihoods)):
        raise ValueError("Os valores devem ser finitos.")
    if np.any(priors < 0) or np.any(likelihoods < 0):
        raise ValueError("Não são permitidos valores negativos.")
    if not np.isclose(priors.sum(), 1.0):
        raise ValueError("Os priors devem somar 1.")

    scores = priors * likelihoods
    evidencia = scores.sum()
    if evidencia == 0:
        raise ZeroDivisionError("A evidence é zero.")
    return scores / evidencia, evidencia, scores

nomes = np.array(["Sensor", "Rede", "Aplicação"])
post_mult, z, scores = atualizar_multiplas(
    [0.50, 0.30, 0.20],
    [0.10, 0.40, 0.70],
)

for nome, score, post in zip(nomes, scores, post_mult):
    print(f"{nome:10} score={score:.3f} posterior={post:.4f}")
print(f"Evidence Z={z:.3f}; MAP={nomes[np.argmax(post_mult)]}")

assert np.isclose(post_mult.sum(), 1)
assert np.allclose(post_mult, [5/31, 12/31, 14/31])
assert nomes[np.argmax(post_mult)] == 'Aplicação'

## 9. Naive Bayes e estabilidade numérica

O exemplo usa dois atributos binários sob a hipótese de independência condicional. Em escala maior, scores são calculados em log para evitar *underflow*.

In [ ]:
classes = np.array(["spam", "legítima"])
priors_classe = np.array([0.20, 0.80])
likelihoods_atributos = np.array([
    [0.70, 0.75],  # promoção e link, dada a classe spam
    [0.05, 0.10],  # promoção e link, dada a classe legítima
])

log_scores = np.log(priors_classe) + np.log(likelihoods_atributos).sum(axis=1)
log_scores_centrados = log_scores - log_scores.max()
scores_estaveis = np.exp(log_scores_centrados)
posteriores_classe = scores_estaveis / scores_estaveis.sum()

for classe, log_score, post in zip(classes, log_scores, posteriores_classe):
    print(f"{classe:8} log-score={log_score:.6f} posterior={post:.6f}")

assert np.allclose(posteriores_classe, [105/109, 4/109])
assert classes[np.argmax(posteriores_classe)] == "spam"
print("A fatoração é uma hipótese do modelo, não uma propriedade garantida dos atributos.")

## 10. Desafios

1. Troque o prior por 0,001 e confira o posterior após um alerta.
2. Mantenha o prior em 0,01 e reduza o falso positivo para 0,005. O que muda?
3. Calcule o posterior após um resultado negativo.
4. Compare duas atualizações independentes com uma única atualização. Depois explique por que repetir o mesmo alerta não constitui nova evidência.
5. Acrescente uma quarta hipótese à função `atualizar_multiplas` e verifique se os posteriors somam 1.
6. Crie 100 atributos com likelihood 0,01 por classe e compare produto direto com log-score.
7. Faça uma análise de sensibilidade com três valores de sensibilidade e três taxas de falso positivo.
8. Escreva uma conclusão que declare explicitamente o que o posterior não prova.

## Conclusões

- Bayes inverte uma condicional usando prior, likelihood e evidence.
- Frequências naturais tornam a taxa-base e o denominador visíveis.
- Um alerta com alta sensibilidade pode ter posterior moderado quando a hipótese é rara.
- Odds transformam atualizações sequenciais em multiplicações de LRs.
- Evidências dependentes não devem ser multiplicadas como se fossem independentes.
- MAP escolhe a hipótese mais provável, mas custos podem exigir outra ação.
- Análise de sensibilidade expõe dependência de priors e taxas operacionais.

## Próxima etapa

Siga para a Aula 05 — **Variáveis aleatórias, PMF, PDF e CDF**. Ela fornecerá a linguagem de distribuições usada para representar resultados numéricos discretos e contínuos.